# Corporation Favorita | Store Sales Forecasting and Retail Analytics


## Introduction
The project employs time series forecasting to predict store sales for Company Favorita, a large Ecuadorian supermarket retailer. The goal is to create a model that is capable of predicting unit sales for thousands of items sold in various Favorita stores.


## Objective
Understand the data: Gain insights into the store sales data, including store-specific information, product families, promotions, and sales numbers. This understanding will enable the company to make informed business decisions.

Predict store sales: Develop a reliable time series forecasting model that accurately predicts the unit sales for different product families at various Favorita stores. This will help the company optimize inventory management, plan promotions, and improve overall sales performance.



## Hypotheses

Null Hypothesis (H0):  Promotional activities does not have a significant impact on the store sales at Corporation Favorita.
Alternative Hypothesis (H1): Promotional activities have a significant impact on the store sales at Corporation Favorita.



## Analytical Questions :

- Is the train dataset complete (has all the required dates)?
- Which dates have the lowest and highest sales for each year (excluding days the store was closed)?
- Compare the sales for each month across the years and determine which month of which year had the highest sales.
- Did the earthquake impact sales?
- Are certain stores or groups of stores selling more products? (Cluster, city, state, type)?
- Are sales affected by promotions, oil prices, and holidays?
- What analysis can we get from the date and its extractable features?
- Which product family and stores did the promotions affect?
- What is the difference between RMSLE, RMSE, MSE (or why is the MAE greater than all of them)?


> **Reproducibility:** Place the Favorita CSV datasets beside this notebook, or set `FAVORITA_DATA_DIR`. SQL Server is optional for the three reference tables. Run all cells from top to bottom. Results depend on the supplied datasets. The 2017 holdout is a chronological evaluation, not random cross-validation.


### Getting Started


### Dependencies and reproducible environment


In [ ]:
# Install only packages missing from your environment, for example:
# %pip install numpy pandas matplotlib seaborn plotly scipy scikit-learn statsmodels python-dotenv pyodbc
# SQL Server connectivity is optional when the CSV files are available.


In [ ]:
"""Core libraries for Favorita sales exploration and forecasting."""
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
from scipy.stats import ttest_ind
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_squared_log_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Keep the project portable: put train.csv, test.csv and transactions.csv here.
DATA_DIR = Path(os.getenv("FAVORITA_DATA_DIR", "."))
RANDOM_SEED = 42
sns.set_theme(style="whitegrid")


# Data Loading
#### Loading the data from SQL database

In [ ]:
# Optional SQL Server connection for the store, oil and holiday tables.
# Store credentials in .env; never commit that file to version control.
try:
    import pyodbc
    from dotenv import dotenv_values
    database_config = dotenv_values(".env")
    required_keys = ("SERVER", "DATABASE", "USERNAME", "PASSWORD")
    if not all(database_config.get(key) for key in required_keys):
        raise ValueError("SQL credentials missing; using local CSV files instead.")
    sql_connection_string = (
        "DRIVER={SQL Server};SERVER={SERVER};DATABASE={DATABASE};"
        "UID={USERNAME};PWD={PASSWORD};MARS_Connection=yes;"
    ).format(**database_config)
    sql_connection = pyodbc.connect(sql_connection_string)
    print("Connected to the Favorita SQL database.")
except (ImportError, ValueError, Exception) as exc:
    sql_connection = None
    print(f"SQL connection unavailable ({exc}); loading reference tables from CSV files.")

def load_favorita_reference_table(table_name: str) -> pd.DataFrame:
    """Load a Favorita reference table from SQL or its local CSV fallback."""
    if sql_connection is not None:
        # Only allow known table names; avoid interpolating untrusted SQL identifiers.
        if table_name not in {"stores", "oil", "holidays_events"}:
            raise ValueError(f"Unsupported table: {table_name}")
        return pd.read_sql_query(f"SELECT * FROM {table_name}", sql_connection)
    csv_path = DATA_DIR / f"{table_name}.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing {csv_path}. Supply the CSV or configure SQL Server.")
    return pd.read_csv(csv_path)

def load_favorita_csv(filename: str) -> pd.DataFrame:
    """Read a local Favorita CSV with a helpful missing-file error."""
    csv_path = DATA_DIR / filename
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing {csv_path}. Set FAVORITA_DATA_DIR to your dataset folder.")
    return pd.read_csv(csv_path)


In [ ]:
# List available SQL tables when connected; otherwise show local reference files.
if sql_connection is not None:
    sql_query = "SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'dbo'"
    database_tables = pd.read_sql_query(sql_query, sql_connection)
    display(database_tables)
else:
    print("Local reference tables:", [p.name for p in DATA_DIR.glob("*.csv")])


In [ ]:
# Load the stores reference dataset (SQL first, local CSV fallback).
store_metadata = load_favorita_reference_table("stores")
display(store_metadata.head())


In [ ]:
# Load the oil reference dataset (SQL first, local CSV fallback).
oil_prices = load_favorita_reference_table("oil")
display(oil_prices.head())


In [ ]:
# Load the holidays_events reference dataset (SQL first, local CSV fallback).
holiday_events = load_favorita_reference_table("holidays_events")
display(holiday_events.head())


### Train data


In [ ]:
# Load the Corporation Favorita train.csv dataset.
favorita_train = load_favorita_csv("train.csv")
display(favorita_train.head())


#### Test data

In [ ]:
# Load the Corporation Favorita test.csv dataset.
favorita_test = load_favorita_csv("test.csv")
display(favorita_test.head())


#### Transaction data

In [ ]:
# Load the Corporation Favorita transactions.csv dataset.
store_transactions = load_favorita_csv("transactions.csv")
display(store_transactions.head())


## Exploratory Data Analysis (EDA), Data Preprocessing & Cleaning

### Understanding the datasets

A comprehensive analysis of the datasets is provided to obtain insight into the available variables, their distributions, and connections. This step will provide an early understanding of the datasets in order to detect any data quality issues that will be addressed during cleaning and preprocessing.

#### i. Shape of The Datasets

In [ ]:
# Print out the shapes of the train and test datasets
print(f"Train Dataset: {favorita_train.shape}")
print(f"Test Datasets: {favorita_test.shape}")


The train dataset contains 3,000,888 rows and 6 columns while the test dataset contains 28,512 rows and 5 columns.

The train dataset is significantly larger than the test dataset in terms of the number of rows. This is expected, as the train dataset is usually larger to provide sufficient data for model training.

In [ ]:
# Print out the shapes of the other datasets
print("Shapes of Each Dataset:")
print(f"Holiday Events Dataset: {holiday_events.shape}")
print(f"Oil Dataset: {oil_prices.shape}")
print(f"Stores Dataset: {store_metadata.shape}")
print(f"Transactions Dataset: {store_transactions.shape}")

- The Holiday Events dataset contains 350 rows and 6 columns. This dataset provides information about various holidays and events.

- The Oil dataset consists of 1,218 rows and 2 columns. This dataset includes information about the daily price of oil.

- The Stores dataset contains 54 rows and 5 columns. This dataset provides details about different stores, such as their locations, types, and clusters.

- The Transactions dataset contains 83,488 rows and 3 columns. This dataset contains information about the number of transactions made at each store on specific dates.

#### ii. Column Information of The Datasets

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
def describe_dataset_columns(dataset_name: str, dataset: pd.DataFrame) -> None:
    """Print column types and non-null counts for a Favorita dataset."""
    print(f"\n{dataset_name} | {len(dataset):,} records")
    dataset.info(show_counts=True)

describe_dataset_columns("Training sales", favorita_train)
describe_dataset_columns("Forecast test", favorita_test)


- The train dataset contains 3,000,888 entries and 6 columns: 'id', 'date', 'store_nbr', 'family', 'sales', and 'onpromotion'.
- The test dataset contains 28,512 entries and 5 columns: 'id', 'date', 'store_nbr', 'family', and 'onpromotion'.
- The "date" column in both datasets of type object. It needs to be converted to a datetime data type for further analysis.
- As expected, the test dataset does not have the "sales" column. This column is not needed because 'sales' is the variable we want to predict. The goal is to use the trained model to predict or forecast the sales in the test data based on the other available features.

In [ ]:
# Column information of the other datasets
describe_dataset_columns('Holiday events', holiday_events)
print()
describe_dataset_columns('Oil', oil_prices)
print()
describe_dataset_columns('Stores', store_metadata)
print()
describe_dataset_columns('Transactions', store_transactions)

The Holiday Events Dataset:
- The dataset contains 350 entries and 6 columns: 'date', 'type', 'locale', 'locale_name', 'description', and 'transferred'.
- The "date" column in the dataset is of type object. It needs to be converted to a datetime data type for further analysis.

The Oil Dataset:
- The dataset contains 1,218 entries has 2 columns: 'date' and 'dcoilwtico'.
- The "date" column in the dataset is of type object. It needs to be converted to a datetime data type for further analysis.
- The 'dcoilwtico' column has 1,175 non-null values, indicating that there are some missing values in this column.

The Stores dataset:
- The dataset contains 54 entries and 5 columns: 'store_nbr', 'city', 'state', 'type', and 'cluster'.

The Transactions dataset:
- The dataset contains 83,488 entries and 3 columns: 'date', 'store_nbr', and 'transactions'.
- The "date" column in the dataset is of type object. It needs to be converted to a datetime data type for further analysis.

### iii. Transforming the 'date' column to datetime format

In [ ]:
# Normalize dates once across every dataset that includes a date column.
for favorita_frame in (favorita_train, favorita_test, holiday_events, oil_prices, store_transactions):
    favorita_frame["date"] = pd.to_datetime(favorita_frame["date"], errors="raise")


In [ ]:
# Confirm the data type of the 'date' column after transformation
print('Date Column Data Type After Transformation:') 
print('==='*14)
print("Train dataset:", favorita_train['date'].dtype)
print("Test dataset:", favorita_test['date'].dtype)
print("Holiday Events dataset:", holiday_events['date'].dtype)
print("Oil dataset:", oil_prices['date'].dtype)
print("Transactions dataset:", store_transactions['date'].dtype)

The 'date' columns in all datasets are correctly formatted.

### iv. Summary statistics of the datasets

In [ ]:
# Checking for the summary statistics of the datasets 
favorita_datasets = {'train': favorita_train, 'test': favorita_test, 'holiday events': holiday_events, 'oil': oil_prices, 'stores': store_metadata, 'transactions': store_transactions}

for name, data in favorita_datasets.items():
    print(f"{name.capitalize()} dataset summary statistics:")
    print('---'*15)
    print(data.describe())
    print('==='*20)
    print()

### v. Checking for Missing Values in The Datasets

In [ ]:
# Check for missing values in the datasets
favorita_datasets = {'train': favorita_train, 'test': favorita_test, 'holiday events': holiday_events, 'oil': oil_prices, 'stores': store_metadata, 'transactions': store_transactions, }

def show_missing_values(favorita_datasets):
    for name, data in favorita_datasets.items():
        print(f"Missing values in the {name.capitalize()} dataset:")
        print(data.isnull().sum())
        print('===' * 18)
        print()

show_missing_values(favorita_datasets)

Only the 'dcoilwtico' (daily crude oil prices) column in the Oil dataset contains missing values. The column contains 43 missing values.

#### Handling the Missing Values in the 'dcoilwtico' (daily crude oil prices) column of the Oil Dataset.

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
import plotly.express as px

# Visualizing the 'dcoilwtico' column to identify a strategy for handling missing values
fig = px.line(oil_prices, x='date', y='dcoilwtico', title='Trend of Oil Prices Over Time')
fig.update_layout(title='Trend of Oil Prices Over Time', title_x=0.5, xaxis_title='Date', yaxis_title='Oil Price')
fig.update_xaxes(rangeslider_visible=True)
fig.show()

#### Fill missing values in the 'dcoilwtico' column using backfill strategy

In [ ]:
# Fill missing values in the 'dcoilwtico' column using backfill strategy
oil_prices['dcoilwtico'] = oil_prices['dcoilwtico'].fillna(method='backfill')

In [ ]:
# Check for missing values after handling
missing_values_after = oil_prices['dcoilwtico'].isnull().sum()
missing_values_after

In [ ]:
# Visualizing the 'dcoilwtico' column to confirm proper handling of Missing Values
fig = px.line(oil_prices, x='date', y='dcoilwtico')
fig.update_layout(title='Trend of Oil Prices Over Time', title_x=0.5, xaxis_title='Date', yaxis_title='Oil Price')
fig.update_xaxes(rangeslider_visible=True)
fig.show()

The missing values in the 'dcoilwitco' column have been handled and are consistent.

### vi. Checking for the completeness of the 'date' column in the Train Dataset

In [ ]:
# Check the completeness of the train dataset
min_date = favorita_train['date'].min()
max_date = favorita_train['date'].max()
expected_dates = pd.date_range(start=min_date, end=max_date)

missing_dates = expected_dates[~expected_dates.isin(favorita_train['date'])]

if len(missing_dates) == 0:
    print("The train dataset is complete. It includes all the required dates.")
else:
    print("The train dataset is incomplete. The following dates are missing:")
    print(missing_dates)

In [ ]:
# Complete the missing dates in the train dataset
# Create an index of the missing dates as a DatetimeIndex object
missing_dates = pd.Index(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[ns]')

# Create a DataFrame with the missing dates, using the 'date' column
missing_data = pd.DataFrame({'date': missing_dates})

# Concatenate the original train dataset and the missing data DataFrame
# ignore_index=True ensures a new index is assigned to the resulting DataFrame
favorita_train = pd.concat([favorita_train, missing_data], ignore_index=True)

# Sort the DataFrame based on the 'date' column in ascending order
favorita_train.sort_values('date', inplace=True)

Confirm if the train dataset is complete

In [ ]:
# Check the completeness of the train dataset
min_date = favorita_train['date'].min()
max_date = favorita_train['date'].max()
expected_dates = pd.date_range(start=min_date, end=max_date)

missing_dates = expected_dates[~expected_dates.isin(favorita_train['date'])]

if len(missing_dates) == 0:
    print("The train dataset is complete. It includes all the required dates.")
else:
    print("The train dataset is incomplete. The following dates are missing:")
    print(missing_dates)

The missing dates were handled successfully.

### vi. Merging The Train Dataset with the Stores, Transactions, Holiday Events and Oil Dataset

In [ ]:
# Merging the common columns ('store_nbr' and 'date') in the datasets using the inner merge() function
# Merge train_df with stores_df based on 'store_nbr' column
merged_df1 = favorita_train.merge(store_metadata, on='store_nbr', how='inner')

# Merge merged_df1 with transactions_df based on 'date' and 'store_nbr' columns
merged_df2 = merged_df1.merge(store_transactions, on=['date', 'store_nbr'], how='inner')

# Merge merged_df2 with holidays_events_df based on 'date' column
merged_df3 = merged_df2.merge(holiday_events, on='date', how='inner')

# Merge merged_df3 with oil_df based on 'date' column
favorita_sales_enriched = merged_df3.merge(oil_prices, on='date', how='inner')

# View the first five rows of the merged dataset
favorita_sales_enriched.head()

The usage of an inner merge in this time series forecasting project for the firm Favorita aids in data consistency, avoiding missing values, and focusing on essential data for good predictions.

An inner merge type keeps only the rows that have matching values in the given columns. In the context of time series forecasting, it enables us to combine datasets using a similar time index or timestamp. By executing an inner merge, we verify that only rows with corresponding timestamps from both datasets are included in the merged output. This is important for time series forecasting since you need to link data from various sources based on timestamps in order to create a consistent and reliable forecasting model.

An inner merge removes any non-matching timestamps, which may be ineffective for forecasting and may bring errors into the dataset. By focusing on the intersection of the datasets, we may produce a merged dataset containing the information required for accurate time series forecasting.

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
favorita_sales_enriched.isnull().sum()

In [ ]:
# Check the column information of the merged dataset
favorita_sales_enriched.info()

The merged dataset after merging the train dataset with additional datasets contains 322,047 rows and 16 columns. Two columns have been renamed as a result of the merging, type_x and type_y.


In [ ]:
# Check the unique values of the two unknown columns
print("Unique values of 'type_x':")
print(favorita_sales_enriched['type_x'].unique())
print()
print("Unique values of 'type_y':")
print(favorita_sales_enriched['type_y'].unique())

The merged dataset consists of 322,047 non-null observations. Two columns have been renamed as a result of the merging, type_x and type_y.

- The type_x column represents the store type.
- The type_y column represents the holiday type.

In [ ]:
# Renaming the columns with the approapriate names
favorita_sales_enriched = favorita_sales_enriched.rename(columns={"type_x": "store_type", "type_y": "holiday_type"})
favorita_sales_enriched.head()

In [ ]:
# Generate summary statistics and transpose the rows and columns of the resultant DataFrame.
# Transposing flips the DataFrame (the rows become columns and the columns become rows) for better readability
favorita_sales_enriched.describe().T


#### viii. Checking for Missing Values in The Datasets

In [ ]:
# Check for missing values in the merged datasets
missing_values = favorita_sales_enriched.isnull().sum()
missing_values

The merged dataset has no missing values.


#### ix. Checking for Duplicate Values in The Datasets

In [ ]:
# Check for duplicate values in the merged dataset
duplicate_rows_merged = favorita_sales_enriched.duplicated()
duplicate_rows_merged.sum()

In [ ]:
# Check for duplicate values in the test dataset
duplicate_rows_test = favorita_test.duplicated()
duplicate_rows_test.sum()

There are no duplicates in the merged and test datasets

#### x. Save the merged dataset in a new CSV file to be used in PowerBI Visualization

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
favorita_sales_enriched.to_csv('Visualization_Data.csv', index=False)
favorita_sales_enriched.head()

In [ ]:
# Create A Copy of The Merged Dataset
forecast_features = favorita_sales_enriched.copy()
forecast_features.info()

## Univariate, Multivariate and Bivariate Analysis

### Univariate Analysis
#### a. Distribution of the 'sales' variable:

In [ ]:
# Histogram
plt.hist(favorita_sales_enriched['sales'], bins=20)
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.title('Distribution of Sales')
plt.show()

# Boxplot
plt.boxplot(favorita_sales_enriched['sales'])
plt.ylabel('Sales')
plt.title('Boxplot of Sales')
plt.show()

The histogram and boxplot of the 'sales' variable provide insights into its distribution. The histogram shows the frequency distribution of sales values. It reveals that the majority of sales fall within a specific range, represented by the peak in the histogram. However, there are also instances of higher sales values, leading to a right-skewed distribution. This skewness suggests that there are relatively fewer occurrences of very high sales, while the majority of sales are concentrated around lower values. The boxplot further confirms the presence of outliers in the data, as indicated by the points beyond the whiskers.

b. distribution of the 'transactions' variable:

In [ ]:
# Histogram
plt.hist(favorita_sales_enriched['transactions'], bins=20)
plt.xlabel('Transactions')
plt.ylabel('Frequency')
plt.title('Distribution of Transactions')
plt.show()

# Boxplot
plt.boxplot(favorita_sales_enriched['transactions'])
plt.ylabel('Transactions')
plt.title('Boxplot of Transactions')
plt.show()

This histogram provides insights into the distribution of transactions in the dataset. The shape of the histogram indicates that the majority of transactions fall into a specific range, which is evident from the high frequency observed on the left side of the histogram. As the transactions increase, the frequency gradually decreases, forming a right-skewed distribution. This suggests that there are relatively fewer instances of high transaction volumes. Overall, the histogram highlights the presence of a cluster of transactions with a lower frequency, indicating a pattern in the data.

c. Distribution of the 'Daily Oil Price' variable:

In [ ]:
# Histogram
plt.hist(favorita_sales_enriched['dcoilwtico'], bins=20)
plt.xlabel('Oil Price')
plt.ylabel('Frequency')
plt.title('Distribution of Oil Price')
plt.show()

# Boxplot
plt.boxplot(favorita_sales_enriched['dcoilwtico'])
plt.ylabel('Oil Price')
plt.title('Boxplot of Oil Price')
plt.show()

The histogram and boxplot of the 'dcoilwtico' variable reveal insights into its distribution. The histogram displays the frequency distribution of oil prices, indicating the number of occurrences for each price range.

## Bivariate Analysis
#### a. Trend of sales over time.

In [ ]:
# Group the data by date and calculate the total sales
daily_sales = favorita_sales_enriched.groupby('date')['sales'].sum().reset_index()

# Create a time series plot with slider
fig = px.line(daily_sales, x='date', y='sales')
fig.update_xaxes(rangeslider_visible=True)
fig.update_layout(title='Trend of Sales Over Time', title_x=0.5)
fig.show()

The plot shows the trend of sales over time. From the graph, we can observe that the sales exhibit some variations and fluctuations over time. There are periods of both high and low sales, indicating potential seasonality or other factors affecting sales patterns.

b. Trend of Daily Crude oil Prices Over Time


In [ ]:
# Visualizing the 'dcoilwtico' column to confirm if the trend is consistent.
fig = px.line(oil_prices, x='date', y='dcoilwtico')
fig.update_layout(title='Trend of Oil Prices Over Time', title_x=0.5, xaxis_title='Date', yaxis_title='Oil Price')
fig.show()

From observation there is a trend in the oil price over time. Oil prices fell at the end of 2014 and have yet to rebound. Despite occasional volatility, oil prices have been stable since the beginning of 2015. As a result, we may witness a dramatic shift in store sales by late 2014. Looking at unit sales data, this is not immediately evident. Although sales appear to be declining in early 2015, they have been increasing since late 2014. Looking at the unit sales data, this is not readily apparent. Although sales do appear to drop off in the early part of 2015, in late 2014 they are rising. Also, to add oil price drop doesn't seems to have any impact on the sales, as it was seen from the sales plot, there was no relation between oil price dropping on the sales, so we can say that this feature or data is of no importance to us and will not be considered during modelling.

c. Total Count of Sales by Store Type

In [ ]:
# Set the color palette to "viridis"
sns.set_palette("viridis")

# Calculate the total count and total sales per store type
store_type_counts = favorita_sales_enriched['store_type'].value_counts()
store_type_sales = favorita_sales_enriched.groupby('store_type')['sales'].sum()

# Create a bar plot with "viridis" color palette for total count
plt.figure(figsize=(8, 6))
sns.barplot(x=store_type_counts.index, y=store_type_counts.values)
plt.xlabel('Store Type')
plt.ylabel('Count')
plt.title('Total Count by Store Type')
plt.show()

The analysis of the total count of sales by store type provides insights into the sales performance and customer demand across different store types. Store Type D stands out with the highest count of sales, suggesting a strong customer base and popularity of products offered. Store Type C follows with a relatively lower count of sales, indicating a significant customer base as well. On the other hand, Store Types A, B, and E have lower counts, suggesting potential areas for improvement or the need to address competition.

d. Total Amount in Sales by Store Type

In [ ]:
# Order the store types by total sales
store_type_sales = store_type_sales.sort_values(ascending=False)

# Create a bar plot with "viridis" color palette for total sales
plt.figure(figsize=(8, 8))
sns.barplot(x=store_type_sales.index, y=store_type_sales.values, order=store_type_sales.index, palette="viridis")
plt.xlabel('Store Type')
plt.ylabel('Total Sales')
plt.title('Total Sales by Store Type')
plt.show()

The total sales amount varies across different store types. Store Type D has the highest total sales, indicating its significant contribution to the overall sales. Store Type A follows closely behind, demonstrating its substantial sales performance. Store Type C ranks third in terms of total sales, while Store Type B and Store Type E have lower sales amounts. Understanding the variations in sales by store type helps identify the key drivers of revenue and highlights the importance of certain store types in driving overall sales.

e. Average Sales by City

In [ ]:
# Group by city and calculate the average sales
average_sales_by_city = favorita_sales_enriched.groupby('city')['sales'].mean()

# Sort the data by average sales in ascending order
average_sales_by_city = average_sales_by_city.sort_values(ascending=True)

# Define colors for the bar plot using 'viridis' color palette
colors = cm.viridis(np.linspace(0, 1, len(average_sales_by_city)))

# Plot the average sales by city horizontally
plt.figure(figsize=(8, 8))
plt.barh(average_sales_by_city.index, average_sales_by_city.values, color=colors)
plt.xlabel('Average Sales')
plt.ylabel('City')
plt.title('Average Sales by City')
plt.show()

Quito has the highest number of stores, significantly surpassing other cities. Cayambe is the second most populated city in terms of stores followed by Ambato, Daule and Loja. Some cities have a moderate number of stores while others have a lower number of stores. Puyo has the lowest number of sales.

f. Average Sales by State

In [ ]:
# Group by state and calculate the average sales
average_sales_by_state = favorita_sales_enriched.groupby('state')['sales'].mean()

# Sort the data by average sales in descending order
average_sales_by_state = average_sales_by_state.sort_values(ascending=True)

# Plot the average sales by state
plt.figure(figsize=(8, 8))
plt.barh(average_sales_by_state.index, average_sales_by_state.values, color=colors)
plt.xlabel('Average Sales')
plt.ylabel('State')
plt.title('Average Sales by State')
plt.xticks(rotation=45)
plt.show()

Pichincha has the highest number of stores, primarily due to the presence of Quito, the capital city. Guayas is the second most populous state in terms of stores, with Guayaquil being a major city. Santo Domingo de los Tsachilas, Azuay, Manabi, Cotopaxi, Tungurahua, Los Rios, El Oro, Chimborazo, Imbabura, Bolivar, Pastaza, Santa Elena, and Loja have a moderate number of stores.

g. Relationship between sales and transactions.

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
sns.scatterplot(x='transactions', y='sales', data=favorita_sales_enriched)
plt.xlabel('Transactions')
plt.ylabel('Sales')
plt.title('Relationship between Sales and Transactions')
plt.show()

The scatter plot visualizes the relationship between sales and transactions in the dataset. Each data point represents a specific instance with corresponding sales and transaction values. Here are the key insights drawn from the scatter plot:

Clustered Data Points: The majority of data points cluster in the lower sales region, forming a specific concentration. This clustering suggests that there are certain transaction volumes that are consistently associated with particular sales levels. This concentration may indicate a common sales pattern or trend that occurs at specific transaction levels.

Outliers: Several data points deviate from the main cluster and are located at higher sales levels for relatively lower transaction volumes or vice versa. These outliers represent exceptional instances where sales are significantly different from what is typically observed for a given number of transactions. Identifying and understanding these outliers can provide valuable insights into unusual sales scenarios or exceptional business activities.

In summary, the scatter plot provides valuable insights into the relationship between sales and transactions. The clustering of data points around specific sales and transaction levels indicates the presence of common patterns. Additionally, outliers represent exceptional cases that warrant further investigation to understand the factors influencing sales and transactions in unique instances. This analysis can help businesses make informed decisions and devise effective strategies to improve sales performance.

## Multivariate Analysis
#### a. Correlation matrix of numerical variables

In [ ]:
# Select numerical variables for correlation analysis
numerical_vars = ['sales', 'transactions', 'dcoilwtico']

# Compute correlation matrix
corr_matrix = favorita_sales_enriched[numerical_vars].corr()

# Plot heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

Correlation values range from -1 to 1, where -1 represents a perfect negative correlation, 1 represents a perfect positive correlation, and 0 represents no correlation. The table helps us understand how the variables are related to each other, providing valuable insights into their interactions. In this correlation matrix, we can see the correlations between different variables:

Sales and Transactions: There is a weak positive correlation of approximately 0.200 between "Sales" and "Transactions." This suggests that there is a slight positive relationship between the number of transactions and the sales. It implies that when the number of transactions increases, there is a tendency for sales to increase as well, although the correlation is not very strong.

Sales and Dcoilwito (Oil Prices): There is a weak negative correlation of approximately -0.062 between "Sales" and "Dcoilwito" (Oil Prices). This indicates a slight negative relationship between sales and oil prices. It suggests that as oil prices increase, there is a tendency for sales to decrease slightly, though the correlation is not significant.

Transactions and Dcoilwito (Oil Prices): There is a very weak negative correlation of approximately -0.017 between "Transactions" and "Dcoilwito" (Oil Prices). This suggests that there is almost no relationship between the number of transactions and oil prices. It indicates that fluctuations in oil prices do not have a significant impact on the number of transactions.

Overall, the correlation values are relatively low, indicating that the relationships between these variables are not very strong. Other factors not considered in this correlation matrix may also influence sales, transactions, and oil prices. It's essential to explore additional factors to gain a more comprehensive understanding of their impact on sales and transactions.

b. Scatter Plot Marrix of numerical Variables

In [ ]:
# Select numerical variables for correlation analysis
numerical_vars = ['sales', 'transactions', 'dcoilwtico']

# Plot scatter plot matrix
sns.pairplot(favorita_sales_enriched[numerical_vars])
plt.show()

The observations of the scatter plot matrix corroborate the observations from the correlation matrix.

### Hypothesis Testing

- Null Hypothesis (H0): The promotional activities have a significant impact on store sales for Corporation Favorita.

- Alternative Hypothesis (H1): The promotional activities have a significant impact on store sales for Corporation Favorita.

In [ ]:
# Welch's t-test compares mean sales for rows with versus without promotion.
# Note: this is an association test, not proof that promotions caused a sales change.
promoted_sales = favorita_sales_enriched.loc[favorita_sales_enriched["onpromotion"] > 0, "sales"].dropna()
unpromoted_sales = favorita_sales_enriched.loc[favorita_sales_enriched["onpromotion"] == 0, "sales"].dropna()
if len(promoted_sales) > 1 and len(unpromoted_sales) > 1:
    promotion_t_statistic, promotion_p_value = ttest_ind(
        promoted_sales, unpromoted_sales, equal_var=False
    )
    print(f"Promotion vs. non-promotion | t={promotion_t_statistic:.3f}, p={promotion_p_value:.4g}")
    print("Reject equal-mean null hypothesis" if promotion_p_value < 0.05 else "Insufficient evidence to reject equal means")
else:
    print("Insufficient observations for a promotion comparison.")


Based on the hypothesis test, we obtained a very low p-value of 0.0. This indicates strong evidence to reject the null hypothesis. Therefore, we can conclude that promotional activities have a significant impact on store sales for Corporation Favorita. The test statistic of 68.22 also suggests a substantial difference in sales between promotional and non-promotional periods. These results support the notion that promotional activities play a crucial role in driving store sales.

### Answering Analytical Questions

### 1. Is the train dataset complete (has all the required dates)?


In [ ]:
# Check the completeness of the train dataset
min_date = favorita_train['date'].min()
max_date = favorita_train['date'].max()
expected_dates = pd.date_range(start=min_date, end=max_date)

missing_dates = expected_dates[~expected_dates.isin(favorita_train['date'])]

if len(missing_dates) == 0:
    print("The train dataset is complete. It includes all the required dates.")
else:
    print("The train dataset is incomplete. The following dates are missing:")
    print(missing_dates)

### 2. Which dates have the lowest and highest sales for each year?

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
favorita_sales_enriched['date'] = pd.to_datetime(favorita_sales_enriched['date'])
favorita_sales_enriched['year'] = favorita_sales_enriched['date'].dt.year

lowest_sales_dates = favorita_sales_enriched.groupby('year')['date'].min()
highest_sales_dates = favorita_sales_enriched.groupby('year')['date'].max()

print("Dates with the lowest sales for each year:\n", lowest_sales_dates)
print("="*50)
print("Dates with the highest sales for each year:\n", highest_sales_dates)

### 3. Analyze the impact of the earthquake on sales

In [ ]:
# Assign the variable earthquake_date to the date the earthquake took place (April 16, 2016)
earthquake_date = pd.to_datetime('2016-04-16')

# Filter the sales data before and after the earthquake
sales_before_earthquake = favorita_train[favorita_train['date'] < earthquake_date]['sales']
sales_after_earthquake = favorita_train[favorita_train['date'] > earthquake_date]['sales']

# Set the colormap to viridis
colormap = cm.get_cmap('viridis')

# Plot the sales before and after the earthquake
plt.plot(sales_before_earthquake, color=colormap(0.2), label='Sales Before Earthquake')
plt.plot(sales_after_earthquake, color=colormap(0.8), label='Sales After Earthquake')
plt.xlabel('Time')
plt.ylabel('Sales')
plt.legend()
plt.show()

There was a surge in sales after the eathquake.

### 4. Determine if certain groups of stores sell more products

In [ ]:
# Group by cluster and calculate the average sales
average_sales_by_cluster = favorita_sales_enriched.groupby('cluster')['sales'].mean()

# Group by city and calculate the average sales
average_sales_by_city = favorita_sales_enriched.groupby('city')['sales'].mean()

# Group by state and calculate the average sales
average_sales_by_state = favorita_sales_enriched.groupby('state')['sales'].mean()

# Group by store type and calculate the average sales
average_sales_by_store_type = favorita_sales_enriched.groupby('store_type')['sales'].mean()

# Set the number of bars in each plot
num_bars = len(average_sales_by_cluster)

# Generate the colors using the viridis palette
colors = plt.cm.viridis(np.linspace(0, 1, num_bars))

In [ ]:
# Sort the data by average sales in descending order
average_sales_by_cluster = average_sales_by_cluster.sort_values(ascending=False)

# Plot the average sales by cluster
plt.figure(figsize=(10, 8))
plt.bar(average_sales_by_cluster.index, average_sales_by_cluster.values, color=colors)
plt.xlabel('Cluster')
plt.ylabel('Average Sales')
plt.title('Average Sales by Cluster')

# Set the x-tick labels as integers
plt.xticks(range(1, len(average_sales_by_cluster.index) + 1))
plt.show()


The cluster with the highest number of stores is Cluster 5, followed by Clusters 14, 8, 11 and 12. These clusters have a significantly larger number of stores compared to the others

In [ ]:
# Sort the data by average sales in descending order
average_sales_by_city = average_sales_by_city.sort_values(ascending=True)

# Plot the average sales by city horizontally
plt.figure(figsize=(8, 8))
plt.barh(average_sales_by_city.index, average_sales_by_city.values, color=colors)
plt.xlabel('Average Sales')
plt.ylabel('City')
plt.title('Average Sales by City')
plt.show()

Quito has the highest number of stores, significantly surpassing other cities. Cayambe is the second most populated city in terms of stores followed by Ambato, Daule and Loja. Some cities have a moderate number of stores while others have a lower number of stores. Puyo has the lowest number of sales.

In [ ]:
# Sort the data by average sales in descending order
average_sales_by_state = average_sales_by_state.sort_values(ascending=True)

# Plot the average sales by state
plt.figure(figsize=(8, 6))
plt.barh(average_sales_by_state.index, average_sales_by_state.values, color=colors)
plt.xlabel('State')
plt.ylabel('Average Sales')
plt.title('Average Sales by State')
plt.xticks(rotation=45)
plt.show()

Pichincha has the highest number of stores, primarily due to the presence of Quito, the capital city. Guayas is the second most populous state in terms of stores, with Guayaquil being a major city. Santo Domingo de los Tsachilas, Azuay, Manabi, Cotopaxi, Tungurahua, Los Rios, El Oro, Chimborazo, Imbabura, Bolivar, Pastaza, Santa Elena, and Loja have a moderate number of stores.

In [ ]:
# Plot the average sales by store type
plt.bar(average_sales_by_store_type.index, average_sales_by_store_type.values, color=colors)
plt.xlabel('Store Type')
plt.ylabel('Average Sales')
plt.title('Average Sales by Store Type')
plt.show()


The store type with the highest average sales is type A followed by Type D. Store Type C has the lowest average sales.

### 5. Are sales affected by promotions, oil prices and holidays?

In [ ]:
# Calculate correlations between sales and promotions, oil prices, holidays
corr_sales_promotions = favorita_sales_enriched['sales'].corr(favorita_sales_enriched['onpromotion'])
corr_sales_oil = favorita_sales_enriched['sales'].corr(favorita_sales_enriched['dcoilwtico'])
corr_sales_holidays = favorita_sales_enriched['sales'].corr(favorita_sales_enriched['holiday_type'] == 'Holiday')

# Print the correlation values
print(f"Correlation between Sales and Promotions: {corr_sales_promotions}")
print(f"Correlation between Sales and Oil Prices: {corr_sales_oil}")
print(f"Correlation between Sales and Holidays: {corr_sales_holidays}")

Promotions: There is a positive correlation of approximately 0.42 between sales and promotions. This suggests that promotions have a moderate positive impact on sales. When promotions are running, there is an increased likelihood of higher sales.

Oil Prices: There is a weak negative correlation of approximately -0.06 between sales and oil prices. This indicates that there is a slight negative relationship between sales and oil prices. However, the correlation is close to zero, suggesting that oil prices have minimal impact on sales.

Holidays: There is a very weak negative correlation of approximately -0.04 between sales and holidays. This indicates that there is almost no relationship between sales and holidays. Holidays do not seem to have a significant impact on sales. These insights suggest that promotions have a relatively stronger influence on sales compared to oil prices and holidays. While promotions positively impact sales, oil prices and holidays show minimal or no relationship with sales.

### 6. What analysis can we get from the date and its extractable features?

In [ ]:
# Extracting Date Components
forecast_features['date'] = pd.to_datetime(forecast_features['date'])
forecast_features['year'] = forecast_features['date'].dt.year
forecast_features['month'] = forecast_features['date'].dt.month
forecast_features['day'] = forecast_features['date'].dt.day
forecast_features.head()

In [ ]:
# Set distinct colors for each year
colors = sns.color_palette("husl", n_colors=len(forecast_features['year'].unique()))

# Visualize the Monthly Sales Trend
monthly_total_sales = forecast_features.groupby(['year', 'month'])['sales'].sum().reset_index()
plt.figure(figsize=(10, 6))
sns.lineplot(data=monthly_total_sales, x='month', y='sales', hue='year', palette=colors)
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.xticks(range(1, 13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend(title='Year')
plt.show()

The line plot depicting the monthly sales trend shows variations in sales throughout the year. Sales seem to dip during the middle months of the year (July to September) before rising again in the last quarter (October to December), with the highest sales in December. The year-wise color distinction helps to observe sales patterns for each year.

In [ ]:
# Visualize the Day of the Week Sales Pattern
forecast_features['day_of_week'] = forecast_features['date'].dt.dayofweek
day_of_week_sales = forecast_features.groupby('day_of_week')['sales'].mean().reset_index()
plt.figure(figsize=(8, 6))
sns.barplot(data=day_of_week_sales, x='day_of_week', y='sales')
plt.title('Average Sales by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Average Sales')
plt.xticks(range(7), labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.show()

In [ ]:
# Visualize the Yearly Sales Trend
yearly_total_sales = forecast_features.groupby('year')['sales'].sum().reset_index()
plt.figure(figsize=(8, 6))
sns.barplot(data=yearly_total_sales, x='year', y='sales')
plt.title('Yearly Sales Trend')
plt.xlabel('Year')
plt.ylabel('Total Sales')
plt.show()

From the date column, we can extract components such as the year, month and day. From these, we can examine daily, monthly and annual sale trends. We can also examine and establish whether specific days such as holidays have higher sales compared to others.

From the date column, we can extract components such as the year, month and day. From these, we can examine daily, monthly and annual sale trends. We can also examine and establish whether specific days such as holidays have higher sales compared to others.

### 7. Are there certain product families types that exhibit higher sales performance?


In [ ]:
# Group the data by product family and calculate the total sales
product_family_revenue = favorita_sales_enriched.groupby('family')['sales'].sum().sort_values(ascending=False)

# Select the top 10 product families
top_10_families = product_family_revenue.head(10)

# Plot the relationship between product family and sales for the top 10 families
plt.figure(figsize=(12, 6))
sns.barplot(x=top_10_families.index, y=top_10_families.values, palette='viridis')
plt.xlabel('Product Family')
plt.ylabel('Total Sales')
plt.title('Total Sales by Product Family (Top 10)')
plt.xticks(rotation=45, ha='right')
plt.show()

The graph illustrates the sales performance of the top 10 product families. Grocery I and beverages exhibit the highest sales, indicating their popularity among customers. Produce and cleaning products also demonstrate significant sales, reflecting the importance of fresh produce and household cleaning supplies. Dairy, bread/bakery, poultry, and meats contribute to overall sales, suggesting the demand for essential food items. Personal care and deli products have relatively lower sales but still play a role in the product mix.

### 8. How does the sales trend vary across different store numbers?

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
store_total_sales = favorita_sales_enriched.groupby('store_nbr')['sales'].sum()

plt.figure(figsize=(14, 6))
plt.bar(store_total_sales.index, store_total_sales.values, color=cm.viridis(store_total_sales.values/max(store_total_sales.values)))
plt.xlabel('Store Number')
plt.ylabel('Total Sales')
plt.title('Sales by Store Number')

# Set the X-axis limits and ticks
plt.xlim(0, 54)
plt.xticks(range(55))

plt.show()

### 9.Which product family and stores did the promotions affect?

### 10.What is the difference between RMSLE, RMSE, MSE (or why is the MAE greater than all of them)?

## Feature Enginering

#### Train Dataset

In [ ]:
# Extracting Date Components
forecast_features['date'] = pd.to_datetime(forecast_features['date'])
forecast_features['year'] = forecast_features['date'].dt.year
forecast_features['month'] = forecast_features['date'].dt.month
forecast_features['day'] = forecast_features['date'].dt.day
forecast_features.head()

#### Dropping Unwanted Columns in The Merged and Test Datasets as it is not needed for our analysis

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
columns_to_drop = ['date','id', 'locale', 'locale_name', 'description', 'store_type', 'transferred', 'state']
forecast_features = forecast_features.drop(columns=columns_to_drop)

forecast_features.head()

#### Product Categorization Based on Families

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
unique_families = forecast_features['family'].unique()
unique_families

In [ ]:
# Consolidate related product families with a single vectorized mapping.
product_family_groups = {
    "FOODS": ["BEVERAGES", "BREAD/BAKERY", "FROZEN FOODS", "MEATS", "PREPARED FOODS", "DELI", "PRODUCE", "DAIRY", "POULTRY", "EGGS", "SEAFOOD"],
    "HOME": ["HOME AND KITCHEN I", "HOME AND KITCHEN II", "HOME APPLIANCES"],
    "CLOTHING": ["LINGERIE", "LADYSWARE"],
    "GROCERY": ["GROCERY I", "GROCERY II"],
    "STATIONERY": ["BOOKS", "MAGAZINES", "SCHOOL AND OFFICE SUPPLIES"],
    "CLEANING": ["HOME CARE", "BABY CARE", "PERSONAL CARE"],
    "HARDWARE": ["PLAYERS AND ELECTRONICS", "HARDWARE"],
}
family_to_category = {
    family: category
    for category, families in product_family_groups.items()
    for family in families
}
forecast_features["family"] = forecast_features["family"].replace(family_to_category)
display(forecast_features.head())


## Modeling

### Data Splitting

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
historical_training_data = encoded_forecast_features.loc[encoded_forecast_features['year'].isin([2013, 2014, 2015, 2016])]
holdout_2017_data = encoded_forecast_features.loc[encoded_forecast_features['year'] == 2017]

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.

historical_training_data.shape

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
holdout_2017_data.head()

In [ ]:
# Favorita retail sales analysis: preserve the original calculation below.
holdout_2017_data.shape

In [ ]:
# Separate the target variable and features for training and testing
train_features = historical_training_data.drop('sales', axis=1)
train_sales_target = historical_training_data['sales'] 


validation_features = holdout_2017_data.drop('sales', axis=1)  
validation_sales_target = holdout_2017_data['sales']

In [ ]:
# Initialize the results dataframe
results_df = pd.DataFrame(columns=['Model', 'RMSLE', 'RMSE', 'MSE', 'MAE'])

## Model 1. Linear Regression

In [ ]:
# Linear regression benchmark for the 2017 holdout period.
linear_sales_model = LinearRegression()
linear_sales_model.fit(train_features, train_sales_target)
linear_sales_predictions = np.clip(linear_sales_model.predict(validation_features), 0, None)

def evaluate_sales_forecast(model_name: str, actual_sales, predicted_sales) -> pd.DataFrame:
    """Return standard regression metrics with valid nonnegative RMSLE inputs."""
    actual = np.asarray(actual_sales, dtype=float)
    predicted = np.clip(np.asarray(predicted_sales, dtype=float), 0, None)
    mse = mean_squared_error(actual, predicted)
    return pd.DataFrame([{
        "Model": model_name,
        "RMSLE": np.sqrt(mean_squared_log_error(np.clip(actual, 0, None), predicted)),
        "RMSE": np.sqrt(mse),
        "MSE": mse,
        "MAE": mean_absolute_error(actual, predicted),
    }])

linear_regression_metrics = evaluate_sales_forecast(
    "Linear Regression", validation_sales_target, linear_sales_predictions
)
display(linear_regression_metrics.round(4))


The Linear Regression model has a relatively low RMSLE, indicating it predicts the logarithm of the target variable well.

The RMSE and MSE values suggest the model has moderate error rates, with larger errors being penalized.

The MAE indicates the average error is 0.37, which is relatively small and suggests good overall accuracy.

## Model 2. Random Forest Regression

In [ ]:
# Random forest benchmark; use parallel CPU workers for faster fitting.
forest_sales_model = RandomForestRegressor(
    n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1
)
forest_sales_model.fit(train_features, train_sales_target)
forest_sales_predictions = forest_sales_model.predict(validation_features)
random_forest_metrics = evaluate_sales_forecast(
    "Random Forest", validation_sales_target, forest_sales_predictions
)
forecast_model_comparison = pd.concat(
    [linear_regression_metrics, random_forest_metrics], ignore_index=True
).sort_values("RMSLE")
display(forecast_model_comparison.round(4))


The Random Forest model shows improved performance compared to the previous Linear Regression model.

Lower RMSLE (0.22) and RMSE (0.72) values indicate better predictive accuracy.

The MSE of 0.51 suggests the model has reduced the average of squared errors.

The MAE of 0.24 shows that the average prediction error is relatively low.